# 02 — Exploratory Analysis & Star Schema Preparation

This notebook continues from the cleaned fact data and performs product-level analysis, then creates dimension keys and exports the analytical tables used by the SQL layer.


## 1. Load the cleaned fact table

Run this notebook after `01_data_loading_cleaning.ipynb`. The saved file is the project's processed fact table.


In [ ]:
df = pd.read_csv("../data/processed/Fact_Orders_enriched.csv")


## 2. Product analysis

Inspect the available product categories and compare their average, total, and maximum prices.


In [ ]:
df["Product type"].unique()


In [ ]:
df.groupby("Product type")["Price"].mean()


In [ ]:
df.groupby("Product type")["Price"].agg(["mean", "sum", "max"])


In [ ]:
print("Overall average price:", df["Price"].mean())


## 3. Product dimension key

Create a stable numeric key for each product category using the mapping supplied in the original analysis.


In [ ]:
Product_type_dict = {
    "haircare": 1,
    "skincare": 2,
    "cosmetics": 3,
}

df["Product_type_Key"] = df["Product type"].map(Product_type_dict)


In [ ]:
dim_product = (
    df[
        [
            "Product_type_Key",
            "Product type",
        ]
    ]
    .drop_duplicates()
    .sort_values("Product_type_Key")
    .reset_index(drop=True)
)

dim_product


## 4. Supplier dimension key

Create a numeric supplier key and build the supplier dimension table.


In [ ]:
df["Supplier name"].unique()


In [ ]:
Supplier_name_dict = {
    "Supplier 1": 1,
    "Supplier 2": 2,
    "Supplier 3": 3,
    "Supplier 4": 4,
    "Supplier 5": 5,
}

df["Supplier_name_Key"] = df["Supplier name"].map(Supplier_name_dict)


In [ ]:
dim_Supplier = (
    df[
        [
            "Supplier_name_Key",
            "Supplier name",
            "Product_type_Key",
        ]
    ]
    .drop_duplicates()
    .sort_values("Supplier_name_Key")
    .reset_index(drop=True)
)

dim_Supplier


## 5. Customer-demographic dimension key

Reapply the supplied `Unknown` → `Male` rule, inspect the categories, map them to numeric keys, and build the customer-demographic dimension.


In [ ]:
df["Customer demographics"] = df["Customer demographics"].replace("Unknown", "Male")
df["Customer demographics"].unique()


In [ ]:
Customer_demographics_dict = {
    "Non-binary": 1,
    "Female": 2,
    "Male": 3,
}

df["Customer_demographics_Key"] = df["Customer demographics"].map(
    Customer_demographics_dict
)


In [ ]:
dim_Customer_demographics = (
    df[
        [
            "Customer_demographics_Key",
            "Customer demographics",
        ]
    ]
    .drop_duplicates()
    .sort_values("Customer_demographics_Key")
    .reset_index(drop=True)
)

dim_Customer_demographics


## 6. Export the star-schema tables

Save the enriched fact table and the three dimension tables to `data/processed/` so they can be consumed by the SQL analysis.


In [ ]:
output_dir = "../data/processed"

df.to_csv(f"{output_dir}/Fact_Orders_enriched.csv", index=False)

dim_product.to_csv(f"{output_dir}/Dim_product.csv", index=False)
dim_Supplier.to_csv(f"{output_dir}/Dim_Supplier.csv", index=False)
dim_Customer_demographics.to_csv(
    f"{output_dir}/Dim_Customer_demographics.csv",
    index=False,
)

print("Star Schema tables saved successfully.")


## Analysis stage complete

The cleaned fact table has been enriched with product, supplier, and customer-demographic keys, and the resulting tables have been exported for the SQL analysis layer.
